# MIMIC basic usage

Fit MIMIC on a mixed tabular dataset, impute missing values, inspect confidence, and plot the original/preprocessed space against the learned embedding.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src" / "mimic").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mimic import MIMIC

rng = np.random.default_rng(0)
n = 120
age = rng.normal(50, 12, n)
income = 20000 + age * 850 + rng.normal(0, 5000, n)
segment = np.where(age > 50, "older", "younger")
outcome = np.where(income > np.median(income), "yes", "no")
df = pd.DataFrame({"id": range(n), "age": age, "income": income, "segment": segment, "outcome": outcome})
df.loc[rng.choice(n, 10, replace=False), "age"] = np.nan
df.loc[rng.choice(n, 8, replace=False), "outcome"] = np.nan
df.head()

In [ ]:
mimic = MIMIC(
    ignore_columns=["id"],
    regression_columns=["age", "income"],
    classification_columns=["segment", "outcome"],
    # modes: "identity", "direct", "factorised", "joint"
    mode="direct",
    capacity=0.25,
    random_state=0,
)
mimic.fit(df)
embedding = mimic.transform(df)
embedding.shape

In [ ]:
imputed, confidence = mimic.impute(df, return_confidence=True)
imputed.head(), confidence.head()

In [ ]:
fig, axes = mimic.plot(df, color_by="outcome", center="random", random_state=0)